# Anki Vocabulary Export

Paste German words or phrases into **Step 1**, then run all cells top-to-bottom to produce an Anki-ready `.tsv` file.

In [2]:
import os
import json
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
# Load OPENAI_API_KEY from a .env file in the repo root (falls back to system env var)
load_dotenv(dotenv_path=r"../.env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Timestamp used to name output files for this session
str_curr_dttm = datetime.now().strftime("%Y-%m-%d_%H-%M")

output_dir       = r"..\output"
output_tsv_path  = os.path.join(output_dir, f"{str_curr_dttm}_anki_vocab_export.tsv")
output_xlsx_path = os.path.join(output_dir, f"{str_curr_dttm}_full_vocab_export.xlsx")

print(f"Session timestamp : {str_curr_dttm}")
print(f"TSV  output path  : {output_tsv_path}")
print(f"Excel output path : {output_xlsx_path}")

Session timestamp : 2026-02-24_10-23
TSV  output path  : ..\output\2026-02-24_10-23_anki_vocab_export.tsv
Excel output path : ..\output\2026-02-24_10-23_full_vocab_export.xlsx


## Step 1 — Paste your German words / phrases below

In [4]:
raw_text = """
Das Gefängnis
Die Abrechnung
Sich verabschieden
Unangenehm
nebenbei
Die lIste abhaken
Aktuell lese ich das Buch XY
Eigentlich habe ich keinen Hunger, aber
Das Schlagloch in der Straße
Der Tennisplatz
Ein Haus mit zwei Etagen
Der Wolkenkratzer
“In Schuss sein” = in gutem Zustand sein
Die Börse
Einen Fehler beheben
Angst überwinden
“Ins kalte Wasser springen"
Obwohl
Der Spielplatz
Das Sofa ist bequem. Das Zimmer ist gemütlich. Und ich fühle mich wohl!
Nicht-fiktionales Texten
Nicht-fiktionaler Text  = Sachtext

 
"""

## Step 2 — Process & enrich via OpenAI

In [5]:
def clean_text(raw_text):
    """Splits a multi-line string into a clean list of non-empty phrases."""
    return [line.strip() for line in raw_text.splitlines() if line.strip()]


def get_vocabulary_data(phrases_list):
    """Calls the OpenAI API to get structured vocabulary data for a list of German phrases."""

    system_prompt = """
    You are an expert German language tutor and data processor. The user will provide a list of German phrases or words.
    Your task is to process each item and return a single JSON object. The JSON object should contain one key,
    "vocabulary", which is a list of objects. Each object must contain the following five keys:
    "deutsch", "deutsch_mit_artikel", "englisch", "afrikaans", and "hinweise".

    - `deutsch`: The original German phrase.
    - `deutsch_mit_artikel`: If the phrase is a noun, prefix it with its article (e.g. "Kürbis" -> "der Kürbis").
      For full sentences or non-nouns, this can be the same as `deutsch`.
    - `englisch`: The English translation.
    - `afrikaans`: The Afrikaans translation.
    - `hinweise`: Part of speech (Wortart), gender (Genus) for nouns, and any other helpful notes.

    Return only the JSON object — no extra text, explanations, or markdown.
    """

    print(f"Sending {len(phrases_list)} phrase(s) to the API...")
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": "\n".join(phrases_list)},
            ],
            response_format={"type": "json_object"},
        )
        data = json.loads(response.choices[0].message.content)
        print("API response received successfully.")
        return data["vocabulary"]
    except Exception as e:
        print(f"API error: {e}")
        return None

In [6]:
df = None

# If an Excel file for this session already exists, load it instead of calling the API again.
# To force a fresh API call, delete (or rename) the cached .xlsx file.
if os.path.exists(output_xlsx_path):
    print(f"✅ Cached file found — loading from '{output_xlsx_path}'.")
    print("   Delete this file to force a fresh API call.")
    df = pd.read_excel(output_xlsx_path)
else:
    phrases = clean_text(raw_text)
    print(f"Phrases to process: {len(phrases)}")

    vocabulary_list = get_vocabulary_data(phrases)

    if vocabulary_list:
        df = pd.DataFrame(vocabulary_list).rename(columns={
            "deutsch":             "Deutsch",
            "deutsch_mit_artikel": "Deutsch mit Artikel",
            "englisch":            "Englisch",
            "afrikaans":           "Afrikaans",
            "hinweise":            "Wortart / Genus / Hinweise",
        })
        df.to_excel(output_xlsx_path, index=False)
        print(f"✅ Full vocabulary saved: {output_xlsx_path}")

# Build the two-column Anki export
if df is not None and not df.empty:
    df["Front"] = df["Deutsch mit Artikel"]
    df["Back"]  = df["Englisch"].fillna("") + " — " + df["Wortart / Genus / Hinweise"].fillna("")

    anki_df = df[["Front", "Back"]]
    anki_df.to_csv(output_tsv_path, sep="\t", index=False, header=False, encoding="utf-8")
    print(f"✅ Anki TSV export ready: {output_tsv_path}")
else:
    print("❌ No data to export.")

Phrases to process: 22
Sending 22 phrase(s) to the API...
API response received successfully.
✅ Full vocabulary saved: ..\output\2026-02-24_10-23_full_vocab_export.xlsx
✅ Anki TSV export ready: ..\output\2026-02-24_10-23_anki_vocab_export.tsv


## Results

In [7]:
df

,Deutsch,Deutsch mit Artikel,Englisch,Afrikaans,Wortart / Genus / Hinweise,Front,Back
0,Das Gefängnis,das Gefängnis,The prison,Die tronk,"Substantiv, Neutrum",das Gefängnis,"The prison — Substantiv, Neutrum"
1,Die Abrechnung,die Abrechnung,The settlement/billing,Die afrekening,"Substantiv, Femininum",die Abrechnung,"The settlement/billing — Substantiv, Femininum"
2,Sich verabschieden,Sich verabschieden,To say goodbye,Afskeid neem,Reflexives Verb,Sich verabschieden,To say goodbye — Reflexives Verb
3,Unangenehm,Unangenehm,Unpleasant,Onaangenaam,Adjektiv,Unangenehm,Unpleasant — Adjektiv
4,nebenbei,nebenbei,Incidentally/Besides,Terloops,Adverb,nebenbei,Incidentally/Besides — Adverb
5,Die lIste abhaken,Die lIste abhaken,To check off the list,Die lys aftik,Ausdruck,Die lIste abhaken,To check off the list — Ausdruck
6,Aktuell lese ich das Buch XY,Aktuell lese ich das Buch XY,"Currently, I'm reading the book XY",Op die oomblik lees ek die boek XY,Aussage/Satz,Aktuell lese ich das Buch XY,"Currently, I'm reading the book XY — Aussage/Satz"
7,"Eigentlich habe ich keinen Hunger, aber","Eigentlich habe ich keinen Hunger, aber","Actually, I'm not hungry, but","Eintlik is ek nie honger nie, maar",Aussage/Satz,"Eigentlich habe ich keinen Hunger, aber","Actually, I'm not hungry, but — Aussage/Satz"
8,Das Schlagloch in der Straße,das Schlagloch in der Straße,The pothole in the street,Die slaggat in die straat,"Substantiv, Neutrum",das Schlagloch in der Straße,"The pothole in the street — Substantiv, Neutrum"
9,Der Tennisplatz,der Tennisplatz,The tennis court,Die tennisbaan,"Substantiv, Maskulinum",der Tennisplatz,"The tennis court — Substantiv, Maskulinum"


In [15]:
anki_df

,Front,Back
0,die Rückmeldung,"feedback — Noun, feminine"
1,der Nationalsozialistische Untergrund,"National Socialist Underground — Noun, masculi..."
2,"sie hat gesprochen, aber sie hat nichts gesagt","she spoke, but she said nothing — Full sentence"
3,die Nagelbomben,"nail bombs — Noun, plural"
4,jemanden ausfragen,"to interrogate someone — Verb, separable prefix"
...,...,...
60,einen Film auf Deutsch synchronisieren,to dub a movie in German — Phrase
61,ich bin in Gedanken,I am lost in thought — Full sentence
62,ich denke daran,I am thinking about it — Full sentence
63,maßgeschneidert,tailored — Adjective
